<a href="https://colab.research.google.com/github/AleximperV/ThucHanhDeepLearning/blob/main/TienXuLyDuLieu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.read_csv('/content/drive/MyDrive/Du_lieu_diem_thi/dulieuxettuyendaihoc.csv')

In [ ]:
# Kiểm tra tổng số lượng dữ liệu thiếu ở mỗi cột
missing_data = df.isnull().sum()

# Chỉ hiển thị những cột có dữ liệu thiếu
print(missing_data[missing_data > 0])

DT    97
dtype: int64


In [ ]:
df_cleaned = df.drop(columns=['DT'])

In [ ]:
# Kiểm tra lại sau khi xử lý
print("Số lượng missing values sau xử lý: ", df_cleaned.isnull().sum().max())

Số lượng missing values sau xử lý:  0


In [ ]:
# Chọn các cột dữ liệu số (bỏ qua cột STT)
num_cols = df.select_dtypes(include=['float64', 'int64']).drop(columns=['STT'], errors='ignore')

# Định nghĩa hàm đếm số lượng dị biệt bằng phương pháp IQR
def count_outliers(col):
    Q1 = col.quantile(0.25)
    Q3 = col.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    # Trả về tổng số dòng lớn hơn upper_bound hoặc nhỏ hơn lower_bound
    return ((col < lower_bound) | (col > upper_bound)).sum()

# Áp dụng hàm trên cho toàn bộ các cột số
outliers_count = num_cols.apply(count_outliers)

# Chỉ in ra những cột có chứa dị biệt (số lượng > 0)
print("Các cột có dị biệt và số lượng tương ứng:")
print(outliers_count[outliers_count > 0])

Các cột có dị biệt và số lượng tương ứng:
L1     1
S1     1
V1     3
D1     2
N1     1
L2     1
L3     1
S3     2
V3     1
X3     1
L4     1
L5     1
S5     2
L6     1
H6     2
S6     2
DH2    4
DH3    2
dtype: int64


In [ ]:
import pandas as pd
import numpy as np

# 1. Đọc dữ liệu
df = pd.read_csv('/content/drive/MyDrive/Du_lieu_diem_thi/dulieuxettuyendaihoc.csv')

# 2. Lấy danh sách các cột là số (bỏ cột STT)
num_cols = df.select_dtypes(include=['float64', 'int64']).columns.drop('STT', errors='ignore')

# 3. Thay thế dị biệt cho từng cột
for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Hàm .clip() sẽ tự động giới hạn các giá trị nằm ngoài biên về lại bằng biên
    df[col] = df[col].clip(lower=lower_bound, upper=upper_bound)

# 4. Lưu lại thành một file mới (để không ghi đè mất file gốc)
df.to_csv('dulieuxettuyendaihoc_no_outliers.csv', index=False)
print("Thay thế thành công và đã lưu file: dulieuxettuyendaihoc_no_outliers.csv")

Thay thế thành công và đã lưu file: dulieuxettuyendaihoc_no_outliers.csv


In [ ]:
import pandas as pd

# 1. Đọc dữ liệu đã được làm sạch từ bước trước
df = pd.read_csv('dulieuxettuyendaihoc_no_outliers.csv')

print("Số lượng các khối thi TRƯỚC khi cân bằng:")
print(df['KT'].value_counts())

# 2. Loại bỏ các cột không cần thiết hoặc chứa quá nhiều missing value (ví dụ cột DT)
if 'DT' in df.columns:
    df = df.drop(columns=['DT'])

# 3. Tìm số lượng của nhóm lớn nhất (ở đây là khối A với 49 dòng)
max_size = df['KT'].value_counts().max()

# 4. Thực hiện Oversampling: Nhân bản dữ liệu của các nhóm nhỏ cho bằng nhóm lớn nhất
lst = [df]
for class_index, group in df.groupby('KT'):
    # Lấy mẫu ngẫu nhiên có hoàn lại (replace=True) để bù đắp phần thiếu hụt
    lst.append(group.sample(max_size - len(group), replace=True, random_state=42))

df_balanced = pd.concat(lst)

# Trộn ngẫu nhiên lại các dòng dữ liệu (tùy chọn nhưng khuyến khích làm)
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print("\nSố lượng các khối thi SAU khi cân bằng:")
print(df_balanced['KT'].value_counts())

# 5. Lưu kết quả ra file mới
df_balanced.to_csv('dulieuxettuyendaihoc_balanced.csv', index=False)
print("\nHoàn tất! Đã lưu thành file: dulieuxettuyendaihoc_balanced.csv")

Số lượng các khối thi TRƯỚC khi cân bằng:
KT
A     49
D1    22
C     14
B      9
A1     6
Name: count, dtype: int64

Số lượng các khối thi SAU khi cân bằng:
KT
C     49
B     49
A1    49
D1    49
A     49
Name: count, dtype: int64

Hoàn tất! Đã lưu thành file: dulieuxettuyendaihoc_balanced.csv
